# Normalizing flows

This notebook is meant as tutorial for normalizing flows (masked autoregressive flows and neural spline flows) as part of the paper "A comparison of generative deep learning methods for multivariate angular simulation". We start by importing some modules.

In [ ]:
import h5py
import numpy as np
import pandas as pd

# The models are implemented in the deep learning framework pytorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch import Tensor
from torch.distributions import Transform, constraints
from torch.utils.data import DataLoader, TensorDataset

# We use the zuko library for the models. This can be installed with !pip install zuko.
import zuko
from zuko.flows import Flow, UnconditionalTransform

from tqdm import tqdm


We will apply our model to the sparse gaussian double Pareto example from the paper. We start by reading in the data. We choose the 10 dimensional case with 10 000 samples:

In [ ]:
with h5py.File("data/sparse_gaussian_data_double_pareto.h5", "r") as h5file:
    print(h5file.keys())
    data = pd.DataFrame(h5file["dataset_n_10000_d_10"][:]).transpose().to_numpy()
    

As we are only interested in the angular part we start by transforming our data into generalized spherical coordinates. We also split it up into a training (80% of the data) and validation set (20% of the data). We will use the validation set to determine when to stop the training:

In [ ]:
def cartesian_to_polar(points):
    """
    Vectorized function to convert multiple points from N-dimensional Cartesian
    coordinates to generalized polar coordinates.

    Parameters:
    points (numpy array): Cartesian coordinates (M points in N-dimensional space).
                          Shape = (M, N)

    Returns:
    numpy array: Polar coordinates for each point.
                 Shape = (M, N), where the first column is the radial distance
                 and the remaining columns are the angles.
    """
    points = np.atleast_2d(points)  # Ensure points are at least 2D
    M, N = points.shape  # M is number of points, N is the dimensionality

    # Step 1: Calculate the radial distance (norm for each point)
    r = np.linalg.norm(points, axis=1)  # Radial distances

    # Step 2: Calculate the angular coordinates
    phi = np.zeros((M, N - 1))
    for i in range(N - 1):
        # Norm of the remaining components starting from index i
        norm = np.linalg.norm(points[:, (i + 1) :], axis=1)
        phi[:, i] = np.arctan2(norm, points[:, i])

    # Special case: The last angle is based on the last two coordinates using arctan2
    phi[:, -1] = np.arctan2(points[:, -1], points[:, -2])

    return r, phi

_, angles = cartesian_to_polar(data)
angles_train = angles[: int(angles.shape[0] * 0.8), :]
angles_val = angles[int(angles.shape[0] * 0.8) :, :]

Next we define the model. For that we first need to implement the last normalizing flow transformation, circularly wrapping the last $d-1$ angle around $[-\pi, \pi]$, whilst using a Gaussian CDF transform for the first $d-2$ angles:

In [ ]:
def gaussian_cdf(x):
    return 0.5 * (1 + torch.erf(x / np.sqrt(2)))

def gaussian_quantile(p):
    return np.sqrt(2) * torch.erfinv(2 * p - 1)

class PhiAndCircular(Transform):
    r"""Creates a transformation that maps the first n-1 dimensions onto the interval [0, pi],
    shifts by 1/2, and circularly maps the last dimension to [-pi, pi]."""

    bijective = True  # This transformation is invertible

    def __init__(self, dim, **kwargs):
        super().__init__(**kwargs)
        
        # Define bounds for transformation
        self.bound_x1 = torch.tensor(torch.pi)  # Upper bound for first (n-1) dimensions. Lower bound is zero.
        self.bound_x2 = torch.tensor(torch.pi)  # Bound for circular mapping of last dimension (circular around [-self.bound_x2, self.bound_x2]).
        
        self.dim = dim  # Store the input dimension size

        # Define domain constraints (input space)
        self.domain = constraints.stack(
            [*[constraints.real] * (self.dim - 1), constraints.interval(-self.bound_x2, self.bound_x2)]
        )
        
        # Define codomain constraints (output space)
        self.codomain = constraints.stack(
            [
                *[constraints.interval(0, self.bound_x1)] * (self.dim - 1),  # First (n-1) dimensions mapped to [0, pi]
                constraints.interval(-self.bound_x2, self.bound_x2),  # Last dimension mapped to [-pi, pi]
            ]
        )

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}"  # String representation of the class

    def _call(self, x: Tensor) -> Tensor:
        """Forward transformation."""
        x1, x2 = x[:, :-1], x[:, -1].unsqueeze(-1)  # Split input tensor into two parts

        # Transform first (n-1) dimensions using Gaussian CDF scaling to [0, pi]
        x1 = self.bound_x1 * gaussian_cdf(x1)

        # Circularly transform last dimension to [-pi, pi]
        x2 = torch.remainder(x2, 2 * self.bound_x2) - self.bound_x2

        # Concatenate the transformed parts and return
        x = torch.cat([x1, x2], dim=-1)
        return x

    def _inverse(self, y: Tensor) -> Tensor:
        """Inverse transformation."""
        y1, y2 = y[:, :-1], y[:, -1].unsqueeze(-1)  # Split output tensor into two parts

        # Inverse transform for first (n-1) dimensions using Gaussian quantile function
        y1 = y1 / self.bound_x1
        y1 = gaussian_quantile(y1)

        # Ensure last dimension remains in [-pi, pi]
        y2 = torch.remainder(y2, 2 * self.bound_x2) - self.bound_x2

        # Concatenate inverse-transformed parts and return
        y = torch.cat([y1, y2], dim=-1)
        return y

    def log_abs_det_jacobian(self, x: Tensor, y: Tensor) -> Tensor:
        """Compute the log absolute determinant of the Jacobian matrix."""
        
        # Compute log-abs-det-Jacobian for first (n-1) dimensions using Gaussian density transformation
        log_abs_det_jacobian = torch.log(self.bound_x1) - x**2 / 2 - np.log(2 * np.pi) / 2
        
        # The last dimension (circular mapping) has a determinant of 1, so log(Jacobian) is 0
        log_abs_det_jacobian[:, -1] = 0
        return log_abs_det_jacobian


We can then define our normalizing flow model. This is how we would define a neural spline flow:

In [ ]:
dimension = angles_train.shape[1]
nsf_flow = zuko.flows.NCSF(dimension, transforms=10)
flow = Flow(
    transform=[
        UnconditionalTransform(lambda: PhiAndCircular(dimension).inv), 
        *nsf_flow.transform.transforms,
    ],
    base=nsf_flow.base,
)

And this an masked autoregressive flow:

In [ ]:
dimension = angles_train.shape[1]
maf_flow = zuko.flows.MAF(dimension, transforms=8)
flow = Flow(
    transform=[
        UnconditionalTransform(lambda: PhiAndCircular(dimension).inv),
        *maf_flow.transform.transforms,
    ],
    base=maf_flow.base,
)

The number of transformations can be set using the `transforms`-attribute in both cases.

We check whether a GPU is available and move the model onto there. Also, we convert the training and validation data from numpy to pytorch and also move it onto the GPU if is available. For minibatching we define dataloaders. These progressively output randomly sampled batches of data:

In [ ]:
# Device setup
enable_cuda = True
device = torch.device("cuda" if torch.cuda.is_available() and enable_cuda else "cpu")
vf = vf.to(device)

# Move neural networks to GPU (if available)
flow = flow.to(device)

# Convert data to torch tensors
angles_train = torch.tensor(angles_train).float().to(device)
angles_val = torch.tensor(angles_val).float().to(device)

# DataLoader for training sets
batch_size = 256
train_loader = DataLoader(TensorDataset(angles_train), batch_size=batch_size, shuffle=True)


We set a number of training parameters and define the optimizer:

In [ ]:
# Training parameters
max_iter = 500  
patience = 50
min_delta = 0.001
best_val_loss = np.inf
counter = 0

# Loss history
loss_hist = np.array([])
loss_hist_val = np.array([])

# Optimizer
optimizer = torch.optim.Adam(flow.parameters(), lr=3e-4)

Finally, our training loop. This one iterates over the dataset for `max_iter` iterations (epochs) until early stopping. In each step, we iterate through the training dataloader and:

- Compute the negative log-likelihood loss for the normalizing flow model.
- Perform backpropagation and update the model parameters.
- Log the average loss over the epoch.

After each epoch, we evaluate the validation loss and apply early stopping if no improvement is detected for a set number of iterations.

In [ ]:
for it in tqdm(range(max_iter), desc="Training"):  # Iterate over training epochs
    flow.train()  # Set the model to training mode
    batch_losses = []  # Track batch losses

    # Iterate through mini-batches
    for batch in train_loader:
        batch_data = batch[0]  # Extract input data
        optimizer.zero_grad()  # Reset gradients

        # Compute negative log-likelihood loss
        loss = -flow().log_prob(batch_data).mean()

        # Backpropagation and optimizer step if loss is valid
        if not (torch.isnan(loss) or torch.isinf(loss)):
            loss.backward()  # Compute gradients
            optimizer.step()  # Update model parameters
            batch_losses.append(loss.item())  # Store loss

    # Compute and store mean batch loss for this epoch
    epoch_loss = np.mean(batch_losses)
    loss_hist = np.append(loss_hist, epoch_loss)

    # Validation phase
    flow.eval()  # Set model to evaluation mode
    with torch.no_grad():
        epoch_val_loss = -flow().log_prob(angles_val).mean().cpu()  # Compute validation loss
    loss_hist_val = np.append(loss_hist_val, epoch_val_loss)

    # Early stopping check
    if epoch_val_loss < best_val_loss - min_delta:  # Check if validation loss improved
        best_val_loss = epoch_val_loss  # Update best loss
        counter = 0  # Reset early stopping counter
    else:
        counter += 1  # Increment counter

    if counter >= patience:  # Stop training if no improvement for 'patience' epochs
        print(f"Early stopping triggered at iteration {it+1}.")
        break

After the model is trained we can sample from it:

In [ ]:
n_samples = int(1e6)
samples = flow((n_samples,).sample().detach().cpu().numpy()

`samples` now contains samples from the generator in generalized spherical coordinates. We can of course transform them to Euclidean coordinates:

In [ ]:
def polar_to_cartesian(r, phi):
    """
    Vectorized function to convert multiple points from N-dimensional polar coordinates
    back to Cartesian coordinates.

    Parameters:
    polar_coords (numpy array): Polar coordinates (M points in N-dimensional space).
                                Shape = (M, N), where the first column is the radial distance
                                and the remaining columns are the angles.

    Returns:
    numpy array: Cartesian coordinates for each point.
                 Shape = (M, N), where each row corresponds to a point in Cartesian coordinates.
    """
    phi = np.atleast_2d(phi)  # Ensure at least 2D
    M, N = phi.shape  # M is number of points, N is the dimensionality (including radial distance)
    N = N + 1

    # Initialize output array for Cartesian coordinates
    cartesian_coords = np.zeros((M, N))

    # Step 1: Compute the first coordinate x_1
    cartesian_coords[:, 0] = r * np.cos(phi[:, 0])

    # Step 2: Compute the remaining coordinates
    sin_product = np.sin(phi[:, 0])  # Cumulative sin product starts with sin(theta_1)
    for i in range(1, N - 1):
        cartesian_coords[:, i] = r * sin_product * np.cos(phi[:, i])
        sin_product *= np.sin(phi[:, i])  # Update sin product cumulatively

    # Step 3: Compute the last coordinate x_N
    cartesian_coords[:, -1] = r * sin_product  # Last component involves only sin product

    return cartesian_coords


samples_euclidean = polar_to_cartesian(np.ones_like(samples), samples)